# Creating a Pore Network Model

__Author(s):__ Cinar Turhan and Masa Prodanovic

__Last Update:__ Sep. 2025

Copyright © 2026 Digital Porous Media Team. All rights reserved.

---

This notebook demonstrates how to extract the medial axis.

In [1]:
## Install packages if you havent:
# !pip install numpy scikit-image pyvista pyvista[jupyter]

In [ ]:
# Import the required packages
import os
import numpy as np
import skimage 
import pyvista as pv
from pathlib import Path
import sys
sys.path.append('../')
from utils.dataloader import load_sample

download_path = Path("/home/jovyan/work/ls6")

In [3]:
# Define a function to use for plotting:
def plot_sample(sample, mesh_kwargs= {}):
    '''
    This function can be used for binary 3D image samples. It includes some parameters to make the plot look visually
    appealing. PyVista package is used in this function. Isosurfaces keyword in contour parameter is used for plotting 
    the contour line between the binary image values. Since the solid phase is "0" and the pore phase is "1", we plot the
    contour between them by specifying isosurfaces to some value in between those two. It is selected as 0.5 here.
    sample: 3D numpy array.
    mesh_kwargs: If you want to specify some visualization parameters based on PyVista's add_mesh function, add them
    under this name. Kwargs ref:
    (https://docs.pyvista.org/version/stable/api/plotting/_autosummary/pyvista.Plotter.add_mesh.html#pyvista.Plotter.add_mesh)
    '''
    plotter = pv.Plotter(notebook=True, lighting='three lights')
    pyvista_image_object = pv.wrap(sample)
    contours = pyvista_image_object.contour(isosurfaces=[0.5])
    if 'color' not in mesh_kwargs:
        mesh_kwargs['color']=(200 / 255, 181 / 255, 152 / 255)
    plotter.add_mesh(contours, **mesh_kwargs)
    plotter.show()

In [4]:
beadpack, castlegate = [load_sample(sample, cache_dir=download_path) for sample in ['beadpack', 'castlegate']]

Loading cached sample 'beadpack' from data/beadpack.tif
Loading cached sample 'castlegate' from data/castlegate.tif


In [5]:
# Select a subset from the data for easier visualization
beadpack_subset = beadpack[0:150, 0:150, 0:150]
castlegate_subset = castlegate[0:100, 0:100, 0:100]

### 3D Visualization
#### 1. Bead Pack

In [6]:
# Close the boundaries of the image:
beadpack_pad = np.pad(beadpack_subset, ((1, 1), (1, 1), (1, 1)), mode='constant', constant_values=0)

# Visualize
plot_sample(beadpack_pad)

Widget(value='<iframe src="http://localhost:33217/index.html?ui=P_0x7fae79da6390_0&reconnect=auto" class="pyvi…

In [7]:
# Close the boundaries of the image:
castlegate_pad = np.pad(castlegate_subset, ((1, 1), (1, 1), (1, 1)), mode='constant', constant_values=0)

# Visualize
plot_sample(castlegate_pad)

Widget(value='<iframe src="http://localhost:33217/index.html?ui=P_0x7fae63dcf810_1&reconnect=auto" class="pyvi…

### Medial Axis Extraction
#### 1. Bead Pack

In [8]:
# Get medial axis
beadpack_medial_axis = skimage.morphology.skeletonize(beadpack_subset)

# Plot
mesh_kwargs = {'line_width':2, 'style':'wireframe', 'color': 'r'}
plot_sample(beadpack_medial_axis, mesh_kwargs)

Widget(value='<iframe src="http://localhost:33217/index.html?ui=P_0x7faddc730c50_2&reconnect=auto" class="pyvi…

#### Make it Fancier:

In [9]:
plotter = pv.Plotter(notebook=True, lighting='three lights')

pv_beadpack_medial_axis = pv.wrap(beadpack_medial_axis)
contours_ma = pv_beadpack_medial_axis.contour(isosurfaces=[0.5])
plotter.add_mesh(contours_ma , color='r', style='wireframe', line_width=3)

pv_beadpack_sample = pv.wrap(beadpack_pad)
contours_sample = pv_beadpack_sample.contour(isosurfaces=[0.5])


def my_plane_func(normal, origin):
    sliced = contours_sample.slice(normal=normal, origin=origin)
    plotter.add_mesh(contours_sample.clip_closed_surface(normal='-z', origin=origin),
                name='arrows',color = (200 / 255, 181 / 255, 152 / 255))


plotter.add_plane_widget(my_plane_func, normal='z',origin=[0, 0, beadpack_medial_axis.shape[2]])
plotter.show()

Widget(value='<iframe src="http://localhost:33217/index.html?ui=P_0x7faddc155d90_3&reconnect=auto" class="pyvi…

### 2. Sandstone

In [10]:
# Get medial axis
castlegate_medial_axis = skimage.morphology.skeletonize(castlegate_subset)

plotter = pv.Plotter(notebook=True, lighting='three lights')

pv_castlegate_medial_axis = pv.wrap(castlegate_medial_axis)
contours_ma = pv_castlegate_medial_axis.contour(isosurfaces=[0.5])
plotter.add_mesh(contours_ma , color='r', style='wireframe', line_width=2)

pv_castlegate_sample = pv.wrap(castlegate_pad)
contours_sample = pv_castlegate_sample.contour(isosurfaces=[0.5])


def my_plane_func(normal, origin):
    sliced = contours_sample.slice(normal=normal, origin=origin)
    plotter.add_mesh(contours_sample.clip_closed_surface(normal='-z', origin=origin),
                name='arrows',color = (200 / 255, 181 / 255, 152 / 255))


plotter.add_plane_widget(my_plane_func, normal='z',origin=[0, 0, castlegate_medial_axis.shape[2]])
plotter.show()

Widget(value='<iframe src="http://localhost:33217/index.html?ui=P_0x7fae64dbbd10_4&reconnect=auto" class="pyvi…